In [1]:
import torch
from urllib.request import urlopen
from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer

/home/icb/muhammed.dasdelen/miniconda3/envs/dinov2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the model and tokenizer
model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')

In [3]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

context_length = 256

In [4]:
import os
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

save_root = '/lustre/groups/labs/marr/qscd01/projects/cytology_vlm_eval/Results'

cell_abbreviation_mapping = {
    "ABE": "Abnormal eosinophil",
    "ART": "Artefact",
    "BAS": "Basophil",
    "BLA": "Blast",
    "EBO": "Erythroblast",
    "EOS": "Eosinophil",
    "FGC": "Faggott cell",
    "HAC": "Hairy cell",
    "KSC": "Smudge cell",
    "LYI": "Immature lymphocyte",
    "LYT": "Lymphocyte",
    "MMZ": "Metamyelocyte",
    "MON": "Monocyte",
    "MYB": "Myelocyte",
    "NGB": "Band neutrophil",
    "NGS": "Segmented neutrophil",
    "NIF": "Not identifiable",
    "OTH": "Other cell",
    "PEB": "Proerythroblast",
    "PLM": "Plasma cell",
    "PMO": "Promyelocyte"
}


root = '/lustre/groups/labs/marr/qscd01/projects/cytology_vlm_eval/Datasets'
for data in ['Acevedo', 'HiCervix', 'Bone_Marrow_Cyto']:
    data_root = os.path.join(root, data, 'test')
    label_csv = pd.read_csv(os.path.join(data_root, f'{data}_test_labels.csv'))
    labels = label_csv.label.unique()
    if data == 'Bone_Marrow_Cyto':
        labels = [cell_abbreviation_mapping[a] for a in labels]
    prompts = [f'an image of {l}' for l in labels]
    tokenized_prompts = tokenizer(prompts, context_length=context_length).to(device)

    names = []
    predictions = []
    
    for image_name in tqdm(label_csv.image_name):
        image_name = f'{image_name}.jpg'
        image = Image.open(os.path.join(data_root, image_name))
        image_tensor = preprocess(image).unsqueeze(0).to(device)

        with torch.inference_mode():
            image_features, text_features, logit_scale = model(image_tensor, tokenized_prompts)
            logits = (logit_scale * image_features @ text_features.t()).detach().softmax(dim=-1)
            sorted_indices = torch.argsort(logits, dim=-1, descending=True)

            logits = logits.cpu().numpy()
            sorted_indices = sorted_indices.cpu().numpy()
            prediction = labels[sorted_indices[0][0]]
        
        names.append(image_name)
        predictions.append(prediction)

    df = pd.DataFrame({'image_name': names, 'cell_type': predictions})
    output_path = os.path.join(save_root, data, f'{data}_0shot_classification_biomedclip_answers.csv')
    try:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
    except Exception as e:
        print(f"Error creating directory: {e}")
        output_path = os.path.join(f'{data}_0shot_classification_biomedclip_answers.csv')
    df.to_csv(output_path, index=False)


100%|██████████| 979/979 [00:56<00:00, 17.40it/s]


In [6]:
data_root = '/lustre/groups/labs/marr/qscd01/projects/cytology_vlm_eval/Datasets/WBCAtt/test'
label_csv = pd.read_csv(os.path.join(data_root, 'WBCAtt_test_labels.csv'))
tasks = label_csv.columns[2:]

tasks_WBCAtt_0shot_classification = {
    'label': ['Neutrophil', 'Eosinophil', 'Basophil', 'Lymphocyte', 'Monocyte'],
    'cell_size': ['big', 'small'],
    'cell_shape': ['round', 'irregular'],
    'nucleus_shape': [
        'unsegmented-band', 'unsegmented-round', 'segmented-multilobed', 
        'segmented-bilobed', 'irregular', 'unsegmented-indented'
    ],
    'nuclear_cytoplasmic_ratio': ['low', 'high'],
    'chromatin_density': ['densely', 'loosely'],
    'cytoplasm_vacuole': ['no', 'yes'],
    'cytoplasm_texture': ['clear', 'frosted'],
    'cytoplasm_colour': ['light blue', 'blue', 'purple blue'],
    'granule_type': ['small', 'round', 'coarse', 'nil'],
    'granule_colour': ['pink', 'purple', 'red', 'nil'],
    'granularity': ['yes', 'no']
}
prompts_WBCAtt_0shot_classification = {
    'label': 'an image of ',
    'cell_size': 'the cell size is ',
    'cell_shape': 'the cell shape is ',
    'nucleus_shape': 'the nucleus shape is ',
    'nuclear_cytoplasmic_ratio': 'the nuclear cytoplasmic ratio is ',
    'chromatin_density': 'the chromatin density is ',
    'cytoplasm_vacuole': 'the cytoplasm vacuole: ',
    'cytoplasm_texture': 'the cytoplasm texture is ',
    'cytoplasm_colour': 'the cytoplasm color is ',
    'granule_type': 'the granule type is ',
    'granule_colour': 'the granule color is ',
    'granularity': 'the granularity: '
}
dfs =[]
for task in tasks:
    labels = tasks_WBCAtt_0shot_classification[task]
    prompt = prompts_WBCAtt_0shot_classification[task]
    prompts = [prompt+l for l in labels]
    
    tokenized_prompts = tokenizer(prompts, context_length=context_length).to(device)

    names = []
    predictions = []
    
    for image_name in tqdm(label_csv['image_name']):
        image_name = f'{image_name}.jpg'
        image = Image.open(os.path.join(data_root, image_name))
        image_tensor = preprocess(image).unsqueeze(0).to(device)

        with torch.inference_mode():
            image_features, text_features, logit_scale = model(image_tensor, tokenized_prompts)
            logits = (logit_scale * image_features @ text_features.t()).detach().softmax(dim=-1)
            sorted_indices = torch.argsort(logits, dim=-1, descending=True)

            logits = logits.cpu().numpy()
            sorted_indices = sorted_indices.cpu().numpy()
            prediction = labels[sorted_indices[0][0]]
        
        names.append(image_name)
        predictions.append(prediction)

    dfs.append(pd.DataFrame({'image_name': names, task: predictions}))

df = pd.DataFrame({'image_name': names})
for d in dfs:
    df = df.merge(d,on='image_name',how='inner')
output_path = os.path.join(save_root,'WBCAtt','answers','WBCAtt_0shot_classification_biomedclip_answers.csv')
df.to_csv(output_path, index=False)
    


100%|██████████| 250/250 [00:03<00:00, 77.67it/s]


In [14]:
from sklearn.metrics import f1_score
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
cell_abbreviation_mapping = {
    "ABE": "Abnormal eosinophil",
    "ART": "Artefact",
    "BAS": "Basophil",
    "BLA": "Blast",
    "EBO": "Erythroblast",
    "EOS": "Eosinophil",
    "FGC": "Faggott cell",
    "HAC": "Hairy cell",
    "KSC": "Smudge cell",
    "LYI": "Immature lymphocyte",
    "LYT": "Lymphocyte",
    "MMZ": "Metamyelocyte",
    "MON": "Monocyte",
    "MYB": "Myelocyte",
    "NGB": "Band neutrophil",
    "NGS": "Segmented neutrophil",
    "NIF": "Not identifiable",
    "OTH": "Other cell",
    "PEB": "Proerythroblast",
    "PLM": "Plasma cell",
    "PMO": "Promyelocyte"
}
root = '/lustre/groups/labs/marr/qscd01/projects/cytology_vlm_eval/Datasets'
save_root = '/lustre/groups/labs/marr/qscd01/projects/cytology_vlm_eval/Results'
for data in ['Acevedo', 'HiCervix', 'Bone_Marrow_Cyto']:
    data_root = os.path.join(root, data, 'test')
    label_csv = pd.read_csv(os.path.join(data_root, f'{data}_test_labels.csv'))
    if data =='Bone_Marrow_Cyto':
        label_csv['label'] = [cell_abbreviation_mapping[a] for a in label_csv['label']]
    predictions = pd.read_csv(os.path.join(save_root, data, 'answers',f'{data}_0shot_classification_biomedclip_answers.csv'))
    predictions['image_name'] = predictions['image_name'].apply(lambda x: x.split('.')[0])
    merged = pd.merge(label_csv, predictions, on='image_name', how='inner')
    merged = merged.drop_duplicates(subset=['image_name'])
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1_scores = []

    for train_idx, test_idx in skf.split(merged, merged['label']):
        fold = merged.iloc[test_idx]
        f1 = f1_score(fold['label'], fold['cell_type'], average='weighted')
        f1_scores.append(f1)

    mean_f1 = np.mean(f1_scores)
    std_f1 = np.std(f1_scores)

    print(f'{data} F1 score - Mean: {mean_f1:.3f}, Std: {std_f1:.3f}')


Acevedo F1 score - Mean: 0.078, Std: 0.013
HiCervix F1 score - Mean: 0.030, Std: 0.005
Bone_Marrow_Cyto F1 score - Mean: 0.030, Std: 0.007


In [25]:
data_root = '/lustre/groups/labs/marr/qscd01/projects/cytology_vlm_eval/Datasets/WBCAtt/test'
label_csv = pd.read_csv(os.path.join(data_root, 'WBCAtt_test_labels.csv'))
tasks = label_csv.columns[2:]
predictions = pd.read_csv(os.path.join(save_root,'WBCAtt','answers','WBCAtt_0shot_classification_biomedclip_answers.csv'))
predictions['image_name'] = predictions['image_name'].apply(lambda x: x.split('.')[0])

predictions = predictions.rename(columns={task: f"{task}_pred" for task in tasks})
merged = pd.merge(label_csv, predictions, on='image_name', how='inner')
merged = merged.drop_duplicates(subset=['image_name'])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for train_idx, test_idx in skf.split(merged, merged['label']):
    print('Fold {}'.format(len(f1_scores) + 1))
    fold_f1_scores = []
    for task in tasks:
        fold = merged.iloc[train_idx]
        f1 = f1_score(fold[task], fold[f'{task}_pred'], average='weighted')
        fold_f1_scores.append(f1)
        print(f'{task} F1 score: {f1}')
    average_f1 = np.mean(fold_f1_scores)
    print(f'Average F1 score: {average_f1}')
    f1_scores.append(average_f1)


overall_average_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)
print('----------------------------------------------------------------------')
print(f'Overall Average F1 score: {overall_average_f1:.3f}, Std: {std_f1:.3f}')


Fold 1
label F1 score: 0.06694560669456066
cell_size F1 score: 0.3961736334405145
cell_shape F1 score: 0.7420377426964254
nucleus_shape F1 score: 0.06780075187969925
nuclear_cytoplasmic_ratio F1 score: 0.7130640668523677
chromatin_density F1 score: 0.8238297872340425
cytoplasm_vacuole F1 score: 0.013317972350230413
cytoplasm_texture F1 score: 0.5504761904761906
cytoplasm_colour F1 score: 0.06497014368352305
granule_type F1 score: 0.18396687936494655
granule_colour F1 score: 0.1391235632183908
granularity F1 score: 0.45
Average F1 score: 0.3509755281575742
Fold 2
label F1 score: 0.06666666666666667
cell_size F1 score: 0.3614754098360655
cell_shape F1 score: 0.7265625
nucleus_shape F1 score: 0.07761470773098679
nuclear_cytoplasmic_ratio F1 score: 0.7314127423822715
chromatin_density F1 score: 0.795268817204301
cytoplasm_vacuole F1 score: 0.018181818181818184
cytoplasm_texture F1 score: 0.5569436201780416
cytoplasm_colour F1 score: 0.05514893617021276
granule_type F1 score: 0.194169342617